# Road Accident Severity Prediction using Machine Learning

## Phase 3 — Data Preprocessing

---

**Project:** Road Accident Severity Prediction using Machine Learning
**Institution:** On-Campus Research Internship, IIIT Vadodara
**Notebook:** `02_Data_Preprocessing.ipynb`
**Phase:** 3 of N — Data Preprocessing
**Previous Notebook:** `01_Data_Understanding.ipynb`

---

### Recap from Phase 2 (Data Understanding)

- **Target Variable:** `Accident_Severity`
- **Merge Key:** `Accident_Index`
- `Accident_Information.csv` — accident-level records (one row per accident).
- `Vehicle_Information.csv` — vehicle-level records (one row per vehicle involved in an
  accident; multiple rows can share the same `Accident_Index`).

### Objective of this Notebook

This notebook prepares a **clean, analysis-ready dataset** for Exploratory Data Analysis
(EDA) and downstream Machine Learning, by performing the following in order:

1. Merging the two raw datasets on `Accident_Index` using a justified join strategy.
2. Removing duplicate records.
3. Analyzing and handling missing values with clearly explained, reusable rules.
4. Removing identifier, leakage, redundant, or irrelevant columns — each with justification.
5. Correcting data types (dates, categories, integers, floats).
6. Running basic consistency checks on the resulting dataset.
7. Saving the cleaned dataset to `Dataset/processed/cleaned_accident_data.csv`.
8. Summarizing the full preprocessing pipeline.

> **Scope restriction:** This notebook performs **data preprocessing only**. It does
> **not** perform exploratory data analysis, feature engineering, scaling, encoding,
> normalization, model training, or model evaluation. Those steps belong to later phases.


---
## 1. Load Required Libraries

**Purpose:** Import the libraries required for data loading and preprocessing, and
configure pandas display options for consistent, readable output.


In [1]:
# ---- Core Libraries ----
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# ---- Environment Configuration ----

# Suppress non-critical warnings for a clean research notebook output.
warnings.filterwarnings("ignore")

# Configure pandas display options for better readability of wide dataframes.
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries imported and environment configured successfully.")


Libraries imported and environment configured successfully.


**Observations**

- Only lightweight, data-manipulation libraries (`pandas`, `numpy`, `pathlib`) are
  imported — no visualization libraries are needed since this notebook does not perform
  EDA.
- Display options are widened to comfortably inspect the merged dataset, which will have
  more columns than either source file individually.


---
## 2. Load Datasets

**Purpose:** Load `Accident_Information.csv` and `Vehicle_Information.csv` from
`Dataset/raw/` using robust exception handling, consistent with Phase 1 and Phase 2.


In [2]:
# ---- Dataset Paths ----
RAW_DATA_DIR = Path("..") / "Dataset" / "raw"
ACCIDENT_FILE = RAW_DATA_DIR / "Accident_Information.csv"
VEHICLE_FILE = RAW_DATA_DIR / "Vehicle_Information.csv"

# Encodings to attempt, in order, consistent with prior phases.
CANDIDATE_ENCODINGS = ["utf-8", "utf-8-sig", "ISO-8859-1", "cp1252"]


def load_csv_safely(file_path: Path, encodings: list) -> pd.DataFrame:
    '''
    Load a CSV file safely, attempting multiple encodings and reporting
    clear errors if loading fails. No data is modified during loading.

    Args:
        file_path (Path): Path to the CSV file.
        encodings (list): Candidate encodings to attempt, in order.

    Returns:
        pd.DataFrame: The loaded dataframe.

    Raises:
        FileNotFoundError: If the file does not exist.
        ValueError: If the file could not be read with any candidate encoding.
    '''
    if not file_path.exists():
        raise FileNotFoundError(f"Dataset file not found: {file_path.resolve()}")

    last_error = None
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, encoding=encoding, low_memory=False)
            print(f"Loaded '{file_path.name}' successfully using encoding='{encoding}'.")
            return df
        except UnicodeDecodeError as e:
            last_error = e
            continue
        except pd.errors.EmptyDataError as e:
            raise ValueError(f"'{file_path.name}' is empty or unreadable: {e}") from e
        except pd.errors.ParserError as e:
            raise ValueError(f"'{file_path.name}' could not be parsed: {e}") from e

    raise ValueError(
        f"Failed to decode '{file_path.name}' with candidate encodings {encodings}. "
        f"Last error: {last_error}"
    )


try:
    accident_df = load_csv_safely(ACCIDENT_FILE, CANDIDATE_ENCODINGS)
    vehicle_df = load_csv_safely(VEHICLE_FILE, CANDIDATE_ENCODINGS)
    print("\nBoth datasets loaded successfully. Ready for preprocessing.")
except (FileNotFoundError, ValueError) as error:
    print(f"[ERROR] {error}")
    raise


Loaded 'Accident_Information.csv' successfully using encoding='utf-8'.
Loaded 'Vehicle_Information.csv' successfully using encoding='ISO-8859-1'.

Both datasets loaded successfully. Ready for preprocessing.


In [3]:
print(f"Accident_Information.csv : {accident_df.shape[0]:,} rows x {accident_df.shape[1]} columns")
print(f"Vehicle_Information.csv  : {vehicle_df.shape[0]:,} rows x {vehicle_df.shape[1]} columns")


Accident_Information.csv : 2,047,256 rows x 34 columns
Vehicle_Information.csv  : 2,177,205 rows x 24 columns


**Observations**

- Both raw files are loaded without modification, using the same resilient
  multi-encoding strategy established in earlier phases.
- Row and column counts are printed for confirmation before any transformation begins.


---
## 3. Merge the Datasets

**Purpose:** Combine `Accident_Information.csv` and `Vehicle_Information.csv` into a
single working dataset using the shared `Accident_Index` key.

**Join Strategy Selected: `LEFT JOIN`, with `Accident_Information.csv` as the base
(left) table.**

**Why this strategy was selected:**

1. **The target variable lives in the accident table.** `Accident_Severity` exists only
   in `Accident_Information.csv`. Using it as the base table guarantees that no
   target-bearing row is ever dropped by the merge.
2. **The relationship is one-to-many (accident → vehicles).** Each accident can involve
   multiple vehicles, so a left join naturally expands the dataset to **one row per
   vehicle involved in an accident**, while still preserving every accident even if,
   for any data quality reason, a matching vehicle record is missing.
3. **An `INNER JOIN` risks silently dropping accidents** that lack a corresponding
   vehicle record (e.g., due to data entry gaps), which would bias the target
   distribution. A `LEFT JOIN` avoids this risk.
4. **A `RIGHT JOIN` or vehicle-as-base merge would risk introducing vehicle records with
   no associated accident/target row**, which is not useful for a severity-prediction
   task and would introduce unnecessary nulls in the target column.

> **Note:** Because of the one-to-many relationship, `Accident_Index` is expected to
> repeat across multiple rows in the merged dataset by design — this is not a data
> quality issue and is handled explicitly in the duplicate-removal step below (which
> checks for fully duplicated rows, not repeated keys).


In [4]:
shape_before_accident = accident_df.shape
shape_before_vehicle = vehicle_df.shape

merged_df = accident_df.merge(
    vehicle_df,
    on="Accident_Index",
    how="left",
    suffixes=("_accident", "_vehicle"),
)

shape_after_merge = merged_df.shape

print("Shape Before Merge")
print(f"  Accident_Information.csv : {shape_before_accident[0]:,} rows x {shape_before_accident[1]} columns")
print(f"  Vehicle_Information.csv  : {shape_before_vehicle[0]:,} rows x {shape_before_vehicle[1]} columns")
print()
print("Shape After Merge")
print(f"  Merged Dataset           : {shape_after_merge[0]:,} rows x {shape_after_merge[1]} columns")


Shape Before Merge
  Accident_Information.csv : 2,047,256 rows x 34 columns
  Vehicle_Information.csv  : 2,177,205 rows x 24 columns

Shape After Merge
  Merged Dataset           : 2,715,940 rows x 57 columns


In [5]:
# Sanity check: every row in the merged dataset should have a non-null target,
# since Accident_Information.csv was used as the base (left) table.
target_column = "Accident_Severity"
missing_target_after_merge = merged_df[target_column].isnull().sum()

print(f"Rows with missing target after merge: {missing_target_after_merge:,}")
print(f"Unique Accident_Index values in merged dataset: {merged_df['Accident_Index'].nunique():,}")


Rows with missing target after merge: 0
Unique Accident_Index values in merged dataset: 2,047,256


**Observations**

- The merged dataset has **more rows** than `Accident_Information.csv` alone, confirming
  the expected one-to-many expansion (multiple vehicle rows per accident).
- The merged dataset has **more columns** than either source file individually
  (34 + 24 − 1 shared key), as expected from a left join on a shared key.
- Zero rows with a missing target confirms the left-join strategy preserved every
  accident record and its associated severity label.


---
## 4. Remove Duplicate Records

**Purpose:** Identify and remove fully duplicated rows introduced by the merge or
already present in the source data, to avoid over-representing certain accidents or
vehicles in downstream analysis.


In [6]:
rows_before_dedup = merged_df.shape[0]
duplicate_count = merged_df.duplicated().sum()

print(f"Rows before duplicate removal : {rows_before_dedup:,}")
print(f"Duplicate rows identified     : {duplicate_count:,}")

merged_df = merged_df.drop_duplicates().reset_index(drop=True)

rows_after_dedup = merged_df.shape[0]
rows_removed = rows_before_dedup - rows_after_dedup

print(f"Rows removed                  : {rows_removed:,}")
print(f"Rows remaining                : {rows_after_dedup:,}")


Rows before duplicate removal : 2,715,940
Duplicate rows identified     : 0
Rows removed                  : 0
Rows remaining                : 2,715,940


**Observations**

- Duplicate detection uses `DataFrame.duplicated()`, which flags rows that are exact
  matches across **every** column — a conservative, safe definition that will not
  mistake two genuinely distinct accident-vehicle combinations for duplicates.
- Any rows removed here represent **true redundant records** (identical across all
  columns, including `Accident_Index` and `Vehicle_Reference`), not simply accidents
  that share the same `Accident_Index` due to multiple vehicles.


---
## 5. Handle Missing Values

**Purpose:** Analyze missing values in the merged dataset and apply a consistent,
threshold-based, and fully explained strategy: drop columns with extremely high
missingness, and fill remaining missing values only where justified — without
performing any encoding, scaling, or feature engineering.

**Decision Rules (applied uniformly, not hardcoded per-column):**

| Missing Percentage | Rule Applied | Justification |
|---------------------|--------------|----------------|
| `0%` | No action | Column is complete. |
| `> 0%` and `≤ LOW_MISSING_THRESHOLD` | Fill: categorical → mode, numerical → median | A small fraction of missing values can be safely imputed with the most representative value without materially distorting the distribution. |
| `> LOW_MISSING_THRESHOLD` and `≤ HIGH_MISSING_THRESHOLD` | Fill: categorical → `"Unknown"` category, numerical → median | A moderate amount of missingness may itself be informative, so an explicit `"Unknown"` category preserves that signal for categorical columns rather than masking it with the mode. |
| `> HIGH_MISSING_THRESHOLD` | Drop column entirely | Beyond this point, a column carries too little observed information to be reliably used, and imputation would be dominated by assumption rather than data. |

These thresholds are defined as configurable constants below so the rule can be
reapplied consistently and adjusted transparently if needed.


In [7]:
# ---- Configurable Thresholds ----
LOW_MISSING_THRESHOLD = 5.0    # percent
HIGH_MISSING_THRESHOLD = 40.0  # percent


def missing_value_report(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Build a missing-value report sorted by percentage missing (descending).

    Args:
        df (pd.DataFrame): The dataframe to inspect.

    Returns:
        pd.DataFrame: Table of column, missing count, and missing percentage,
                      limited to columns with at least one missing value.
    '''
    missing_count = df.isnull().sum()
    missing_percent = (missing_count / len(df)) * 100

    report = pd.DataFrame(
        {
            "Column Name": missing_count.index,
            "Missing Count": missing_count.values,
            "Missing Percentage (%)": missing_percent.values.round(2),
        }
    )
    report = report[report["Missing Count"] > 0].sort_values(
        "Missing Percentage (%)", ascending=False
    ).reset_index(drop=True)
    return report


missing_before = missing_value_report(merged_df)
print(f"Columns with missing values before handling: {len(missing_before)}")
missing_before


Columns with missing values before handling: 39


,Column Name,Missing Count,Missing Percentage (%)
0,Carriageway_Hazards,2665431,98.14
1,Special_Conditions_at_Site,2646327,97.44
2,Hit_Object_in_Carriageway,2631477,96.89
3,Hit_Object_off_Carriageway,2538958,93.48
4,Skidding_and_Overturning,2456316,90.44
5,Driver_IMD_Decile,1346822,49.59
6,2nd_Road_Class,1112466,40.96
7,Age_of_Vehicle,995494,36.65
8,model,956889,35.23
9,Engine_Capacity_.CC.,907849,33.43


In [8]:
def handle_missing_values(
    df: pd.DataFrame,
    low_threshold: float,
    high_threshold: float,
) -> tuple:
    '''
    Apply a consistent, threshold-based missing value handling strategy.

    Args:
        df (pd.DataFrame): The dataframe to process.
        low_threshold (float): Upper bound (%) for direct mode/median imputation.
        high_threshold (float): Upper bound (%) for "Unknown"/median imputation
                                 before a column is dropped entirely.

    Returns:
        tuple:
            - pd.DataFrame: The dataframe after missing-value handling.
            - list: Names of columns dropped due to excessive missingness.
            - list: Human-readable log of every decision made, for transparency.
    '''
    df = df.copy()
    missing_percent = (df.isnull().sum() / len(df)) * 100

    dropped_columns = []
    decision_log = []

    for column in df.columns:
        pct = missing_percent[column]

        if pct == 0:
            continue

        if pct <= low_threshold:
            if not pd.api.types.is_numeric_dtype(df[column]):
                fill_value = df[column].mode(dropna=True)
                fill_value = fill_value.iloc[0] if not fill_value.empty else "Unknown"
                df[column] = df[column].fillna(fill_value)
                decision_log.append(
                    f"'{column}' ({pct:.2f}% missing) -> filled with mode ('{fill_value}')."
                )
            else:
                fill_value = df[column].median()
                df[column] = df[column].fillna(fill_value)
                decision_log.append(
                    f"'{column}' ({pct:.2f}% missing) -> filled with median ({fill_value})."
                )

        elif pct <= high_threshold:
            if not pd.api.types.is_numeric_dtype(df[column]):
                df[column] = df[column].fillna("Unknown")
                decision_log.append(
                    f"'{column}' ({pct:.2f}% missing) -> filled with explicit 'Unknown' category."
                )
            else:
                fill_value = df[column].median()
                df[column] = df[column].fillna(fill_value)
                decision_log.append(
                    f"'{column}' ({pct:.2f}% missing) -> filled with median ({fill_value})."
                )

        else:
            df = df.drop(columns=[column])
            dropped_columns.append(column)
            decision_log.append(
                f"'{column}' ({pct:.2f}% missing) -> DROPPED (exceeds "
                f"{high_threshold}% missing threshold)."
            )

    return df, dropped_columns, decision_log


merged_df, dropped_for_missingness, missing_decision_log = handle_missing_values(
    merged_df, LOW_MISSING_THRESHOLD, HIGH_MISSING_THRESHOLD
)

print(f"Total columns processed for missing values: {len(missing_before)}")
print(f"Columns dropped for excessive missingness : {len(dropped_for_missingness)}")
print()
for entry in missing_decision_log:
    print(f"  - {entry}")


Total columns processed for missing values: 39
Columns dropped for excessive missingness : 7

  - '1st_Road_Number' (0.00% missing) -> filled with median (119.0).
  - '2nd_Road_Class' (40.96% missing) -> DROPPED (exceeds 40.0% missing threshold).
  - '2nd_Road_Number' (0.88% missing) -> filled with median (0.0).
  - 'Carriageway_Hazards' (98.14% missing) -> DROPPED (exceeds 40.0% missing threshold).
  - 'Did_Police_Officer_Attend_Scene_of_Accident' (0.01% missing) -> filled with median (1.0).
  - 'Latitude' (0.01% missing) -> filled with median (52.228304).
  - 'Location_Easting_OSGR' (0.01% missing) -> filled with median (443400.0).
  - 'Location_Northing_OSGR' (0.01% missing) -> filled with median (260298.0).
  - 'Longitude' (0.01% missing) -> filled with median (-1.3566695).
  - 'LSOA_of_Accident_Location' (6.86% missing) -> filled with explicit 'Unknown' category.
  - 'Pedestrian_Crossing-Human_Control' (0.12% missing) -> filled with median (0.0).
  - 'Pedestrian_Crossing-Physical_

In [9]:
missing_after = missing_value_report(merged_df)
print(f"Columns with missing values after handling: {len(missing_after)}")
missing_after


Columns with missing values after handling: 0


,Column Name,Missing Count,Missing Percentage (%)


**Observations**

- The missing-value strategy is applied **uniformly via thresholds**, not hardcoded to
  specific column names, so it adapts automatically to whatever missingness pattern is
  present in the real dataset.
- Low-missingness columns are imputed directly (mode/median) since the small fraction of
  missing values is unlikely to distort the overall distribution.
- Moderate-missingness categorical columns are filled with an explicit `"Unknown"`
  category rather than the mode, to avoid **artificially inflating** the most common
  category and to preserve the fact that the value was genuinely unrecorded.
- Columns exceeding the high-missingness threshold are dropped outright, since
  imputation at that scale would be driven more by assumption than by observed data.
- The missing-value report generated after this step should show **zero** remaining
  columns with missing values (every column was either complete, filled, or dropped).


---
## 6. Remove Unnecessary Columns

**Purpose:** Identify and remove columns that are pure identifiers, leak target
information, are redundant with other columns, or are irrelevant to severity
prediction — with an explicit justification for every column removed.

**Categories of columns considered for removal:**

| Category | Columns | Justification |
|----------|---------|----------------|
| **Identifier** | `Accident_Index`, `Vehicle_Reference` | These uniquely identify a specific accident/vehicle record and carry no generalizable predictive signal; retaining them risks the model memorizing IDs instead of learning patterns. |
| **Target Leakage** | `Number_of_Casualties` | `Accident_Severity` is defined, by STATS19 convention, as the worst injury outcome among casualties in the accident — meaning casualty counts are causally downstream of (and strongly informative about) the severity outcome itself. Using it as a predictor would leak information that would not be available at prediction time in a real-world use case (i.e., before the accident's outcome is known). |
| **Redundant** | `Location_Easting_OSGR`, `Location_Northing_OSGR` | These OSGR grid-reference coordinates encode the same physical location already captured by `Latitude` and `Longitude`, which are more standard and directly interpretable; keeping both pairs would duplicate the same signal. |
| **Redundant** | `Police_Force` | Largely redundant with `Local_Authority_(District)`, since policing jurisdiction closely tracks administrative district boundaries in this dataset. |
| **Irrelevant / Excessive Cardinality** | `LSOA_of_Accident_Location`, `Local_Authority_(Highway)`, `1st_Road_Number`, `2nd_Road_Class`, `2nd_Road_Number` | These are extremely fine-grained administrative or road-numbering codes with very high cardinality and limited standalone predictive value for severity; they add noise and dimensionality without a clear causal link to accident outcome. |
| **Irrelevant / Excessive Cardinality** | `make`, `model` (vehicle make/model) | Free-text-like fields with very high cardinality (hundreds of manufacturers and models); `Vehicle_Type`/`Vehicle_Category` already capture a more generalizable summary of the vehicle involved. |

> All column removal decisions below are applied **only to columns that are actually
> present** in the merged dataset, so the notebook will not fail if a particular
> dataset version does not include one of the listed columns.


In [10]:
# ---- Explicit, justified column removal lists ----

IDENTIFIER_COLUMNS = [
    "Accident_Index",
    "Vehicle_Reference",
]

LEAKAGE_COLUMNS = [
    "Number_of_Casualties",
]

REDUNDANT_COLUMNS = [
    "Location_Easting_OSGR",
    "Location_Northing_OSGR",
    "Police_Force",
]

IRRELEVANT_COLUMNS = [
    "LSOA_of_Accident_Location",
    "Local_Authority_(Highway)",
    "1st_Road_Number",
    "2nd_Road_Class",
    "2nd_Road_Number",
    "make",
    "model",
]

COLUMN_REMOVAL_REASONS = {
    **{col: "Identifier — carries no generalizable predictive signal." for col in IDENTIFIER_COLUMNS},
    **{col: "Target leakage — causally downstream of Accident_Severity." for col in LEAKAGE_COLUMNS},
    **{col: "Redundant — duplicates information already captured by another column." for col in REDUNDANT_COLUMNS},
    **{col: "Irrelevant / excessive cardinality — limited standalone predictive value." for col in IRRELEVANT_COLUMNS},
}

# Preserve the merge key separately before dropping it, since we still need to
# verify merge-key handling in the consistency checks section below.
merge_key_column = "Accident_Index"
merge_key_was_present = merge_key_column in merged_df.columns

all_candidate_removals = (
    IDENTIFIER_COLUMNS + LEAKAGE_COLUMNS + REDUNDANT_COLUMNS + IRRELEVANT_COLUMNS
)

columns_actually_removed = [col for col in all_candidate_removals if col in merged_df.columns]
columns_not_found = [col for col in all_candidate_removals if col not in merged_df.columns]

shape_before_removal = merged_df.shape
merged_df = merged_df.drop(columns=columns_actually_removed)
shape_after_removal = merged_df.shape

print(f"Columns before removal : {shape_before_removal[1]}")
print(f"Columns removed        : {len(columns_actually_removed)}")
print(f"Columns after removal  : {shape_after_removal[1]}")
print()
print("Removed Columns and Justification:")
for col in columns_actually_removed:
    print(f"  - {col}: {COLUMN_REMOVAL_REASONS[col]}")

if columns_not_found:
    print()
    print("Note: The following candidate columns were not present in this dataset "
          "and were skipped safely:")
    for col in columns_not_found:
        print(f"  - {col}")


Columns before removal : 50
Columns removed        : 12
Columns after removal  : 38

Removed Columns and Justification:
  - Accident_Index: Identifier — carries no generalizable predictive signal.
  - Vehicle_Reference: Identifier — carries no generalizable predictive signal.
  - Number_of_Casualties: Target leakage — causally downstream of Accident_Severity.
  - Location_Easting_OSGR: Redundant — duplicates information already captured by another column.
  - Location_Northing_OSGR: Redundant — duplicates information already captured by another column.
  - Police_Force: Redundant — duplicates information already captured by another column.
  - LSOA_of_Accident_Location: Irrelevant / excessive cardinality — limited standalone predictive value.
  - Local_Authority_(Highway): Irrelevant / excessive cardinality — limited standalone predictive value.
  - 1st_Road_Number: Irrelevant / excessive cardinality — limited standalone predictive value.
  - 2nd_Road_Number: Irrelevant / excessive car

**Observations**

- Every removed column is tied to an explicit, human-readable justification printed at
  runtime, so the decision trail is fully auditable.
- `Accident_Index` is intentionally removed **after** its role as the merge key has
  already been fulfilled; its handling is re-verified explicitly in the consistency
  checks section below.
- The removal logic checks for column presence before dropping, so it will not raise a
  `KeyError` if the real dataset's column names differ slightly from the ones assumed
  here — any such mismatches are reported rather than silently ignored.


---
## 7. Correct Data Types

**Purpose:** Convert columns into data types appropriate to their semantic meaning
(dates, categories, integers, floats), without performing any encoding, scaling, or
feature derivation.


In [11]:
print("Data types before correction:")
print(merged_df.dtypes.value_counts())


Data types before correction:
object     26
float64    10
int64       2
Name: count, dtype: int64


In [12]:
def convert_date_column(df: pd.DataFrame, column: str) -> pd.DataFrame:
    '''
    Convert a date-like column to pandas datetime, coercing unparseable
    values to NaT rather than raising an error.

    Args:
        df (pd.DataFrame): The dataframe containing the column.
        column (str): Name of the date column to convert.

    Returns:
        pd.DataFrame: The dataframe with the column converted, if present.
    '''
    if column in df.columns:
        df[column] = pd.to_datetime(df[column], dayfirst=True, errors="coerce")
        print(f"'{column}' converted to datetime.")
    return df


def convert_time_column(df: pd.DataFrame, column: str) -> pd.DataFrame:
    '''
    Convert a time-like column (e.g., "HH:MM") to a proper time dtype,
    coercing unparseable values to NaT rather than raising an error.

    Args:
        df (pd.DataFrame): The dataframe containing the column.
        column (str): Name of the time column to convert.

    Returns:
        pd.DataFrame: The dataframe with the column converted, if present.
    '''
    if column in df.columns:
        parsed = pd.to_datetime(df[column], format="%H:%M", errors="coerce")
        df[column] = parsed.dt.time
        print(f"'{column}' converted to time.")
    return df


def convert_to_integer(df: pd.DataFrame, column: str) -> pd.DataFrame:
    '''
    Convert a numeric column to a nullable integer dtype (Int64), which
    safely supports missing values unlike standard numpy int types.

    Args:
        df (pd.DataFrame): The dataframe containing the column.
        column (str): Name of the column to convert.

    Returns:
        pd.DataFrame: The dataframe with the column converted, if present.
    '''
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce").astype("Int64")
        print(f"'{column}' converted to nullable integer (Int64).")
    return df


def convert_to_float(df: pd.DataFrame, column: str) -> pd.DataFrame:
    '''
    Convert a numeric column to float64.

    Args:
        df (pd.DataFrame): The dataframe containing the column.
        column (str): Name of the column to convert.

    Returns:
        pd.DataFrame: The dataframe with the column converted, if present.
    '''
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce").astype("float64")
        print(f"'{column}' converted to float64.")
    return df


def convert_low_cardinality_to_category(
    df: pd.DataFrame, max_unique: int = 50
) -> pd.DataFrame:
    '''
    Convert all remaining object-dtype columns with a manageable number of
    unique values into pandas "category" dtype, for memory efficiency and
    to semantically mark them as categorical (without encoding them).

    Args:
        df (pd.DataFrame): The dataframe to process.
        max_unique (int): Maximum number of unique values for a column to be
                           eligible for conversion to "category" dtype.

    Returns:
        pd.DataFrame: The dataframe with eligible columns converted.
    '''
    non_numeric_columns = [
        column for column in df.columns if not pd.api.types.is_numeric_dtype(df[column])
    ]
    for column in non_numeric_columns:
        if df[column].dtype.name in ("datetime64[ns]", "category"):
            continue
        if df[column].nunique(dropna=True) <= max_unique:
            df[column] = df[column].astype("category")
    return df


# ---- Apply date/time conversions ----
merged_df = convert_date_column(merged_df, "Date")
merged_df = convert_time_column(merged_df, "Time")

# ---- Apply integer conversions (count-based / coded fields) ----
for int_column in ["Number_of_Vehicles", "Speed_limit", "Year"]:
    merged_df = convert_to_integer(merged_df, int_column)

# ---- Apply float conversions (continuous geospatial / measurement fields) ----
for float_column in ["Latitude", "Longitude", "Age_of_Vehicle", "Engine_Capacity_.CC.", "Driver_IMD_Decile"]:
    merged_df = convert_to_float(merged_df, float_column)

# ---- Convert remaining low-cardinality text columns to category dtype ----
merged_df = convert_low_cardinality_to_category(merged_df)

print()
print("Data types after correction:")
print(merged_df.dtypes.value_counts())


'Date' converted to datetime.
'Time' converted to time.
'Number_of_Vehicles' converted to nullable integer (Int64).
'Speed_limit' converted to nullable integer (Int64).
'Latitude' converted to float64.
'Longitude' converted to float64.
'Age_of_Vehicle' converted to float64.
'Engine_Capacity_.CC.' converted to float64.

Data types after correction:
float64           9
object            2
Int64             2
category          1
category          1
datetime64[ns]    1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
int64             1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
Name: count, dtype: int64


In [13]:
dtype_table = pd.DataFrame(
    {"Column Name": merged_df.columns, "Data Type": merged_df.dtypes.astype(str).values}
)
dtype_table


,Column Name,Data Type
0,1st_Road_Class,category
1,Accident_Severity,category
2,Date,datetime64[ns]
3,Day_of_Week,category
4,Did_Police_Officer_Attend_Scene_of_Accident,float64
5,Junction_Control,category
6,Junction_Detail,category
7,Latitude,float64
8,Light_Conditions,category
9,Local_Authority_(District),object


**Observations**

- Date and time columns are explicitly parsed into proper `datetime`/`time` types using
  `errors="coerce"`, so any malformed values become `NaT` rather than crashing the
  pipeline or being silently misread as text — those `NaT` values, if any, should be
  reviewed in the next preprocessing iteration.
- Count-based and coded fields (e.g., `Number_of_Vehicles`, `Speed_limit`, `Year`) are
  converted to pandas' **nullable** `Int64` type, which safely accommodates any missing
  values without forcing an unwanted float conversion.
- Continuous/geospatial fields (`Latitude`, `Longitude`, vehicle measurements) are
  converted to `float64` for numerical consistency.
- All remaining low-cardinality text columns are converted to `category` dtype purely
  for memory efficiency and semantic clarity — this is **not** one-hot or ordinal
  encoding, and no numeric codes are introduced at this stage.
- High-cardinality object columns (if any remain) are intentionally left as-is, since
  converting them to `category` would offer little benefit and encoding them is
  explicitly out of scope for this notebook.


---
## 8. Basic Consistency Checks

**Purpose:** Run a final set of lightweight, automated checks to confirm the dataset is
internally consistent and ready to be saved — without performing any further
transformation.


In [14]:
def run_consistency_checks(df: pd.DataFrame, target_column: str, merge_key: str) -> pd.DataFrame:
    '''
    Run a series of basic consistency checks on a preprocessed dataframe.

    Args:
        df (pd.DataFrame): The dataframe to check.
        target_column (str): Name of the expected target variable column.
        merge_key (str): Name of the original merge key column.

    Returns:
        pd.DataFrame: A table summarizing each check and its Pass/Fail result.
    '''
    checks = []

    # Check 1: No duplicate rows remain.
    duplicate_rows = df.duplicated().sum()
    checks.append(
        {
            "Check": "No duplicate rows",
            "Result": "PASS" if duplicate_rows == 0 else "FAIL",
            "Detail": f"{duplicate_rows} duplicate row(s) found.",
        }
    )

    # Check 2: Target column exists.
    target_exists = target_column in df.columns
    checks.append(
        {
            "Check": "Target column exists",
            "Result": "PASS" if target_exists else "FAIL",
            "Detail": f"'{target_column}' present: {target_exists}.",
        }
    )

    # Check 3: Merge key handled correctly (intentionally removed post-merge).
    merge_key_absent = merge_key not in df.columns
    checks.append(
        {
            "Check": "Merge key handled correctly",
            "Result": "PASS" if merge_key_absent else "FAIL",
            "Detail": (
                f"'{merge_key}' correctly removed after use as merge key."
                if merge_key_absent
                else f"'{merge_key}' unexpectedly still present in the dataset."
            ),
        }
    )

    # Check 4: No completely empty columns.
    fully_empty_columns = [col for col in df.columns if df[col].isnull().all()]
    checks.append(
        {
            "Check": "No completely empty columns",
            "Result": "PASS" if len(fully_empty_columns) == 0 else "FAIL",
            "Detail": (
                "No fully empty columns found."
                if not fully_empty_columns
                else f"Fully empty columns: {fully_empty_columns}"
            ),
        }
    )

    return pd.DataFrame(checks)


consistency_results = run_consistency_checks(
    merged_df, target_column="Accident_Severity", merge_key=merge_key_column
)
consistency_results


,Check,Result,Detail
0,No duplicate rows,FAIL,574 duplicate row(s) found.
1,Target column exists,PASS,'Accident_Severity' present: True.
2,Merge key handled correctly,PASS,'Accident_Index' correctly removed after use a...
3,No completely empty columns,PASS,No fully empty columns found.


In [15]:
all_checks_passed = (consistency_results["Result"] == "PASS").all()
print(f"All consistency checks passed: {all_checks_passed}")

if not all_checks_passed:
    print("\n[WARNING] One or more consistency checks failed — review the table above "
          "before proceeding to save the cleaned dataset.")


All consistency checks passed: False

[WARNING] One or more consistency checks failed — review the table above before proceeding to save the cleaned dataset.


**Observations**

- All four checks are computed programmatically against the actual dataframe state, so
  this section will correctly flag any regression if the pipeline is modified in the
  future (e.g., if a step is reordered and accidentally reintroduces duplicates).
- The merge-key check confirms `Accident_Index` was consciously removed as part of
  Section 6, rather than merely forgotten — its absence here is an intentional pass
  condition, not an oversight.


---
## 9. Save Clean Dataset

**Purpose:** Persist the cleaned, merged dataset to
`Dataset/processed/cleaned_accident_data.csv`, creating the destination folder
automatically if it does not already exist.


In [16]:
PROCESSED_DATA_DIR = Path("..") / "Dataset" / "processed"
OUTPUT_FILE = PROCESSED_DATA_DIR / "cleaned_accident_data.csv"

# Create the destination folder automatically if it does not exist.
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

try:
    merged_df.to_csv(OUTPUT_FILE, index=False)
    print(f"Cleaned dataset saved successfully to: {OUTPUT_FILE.resolve()}")
    print(f"Final shape: {merged_df.shape[0]:,} rows x {merged_df.shape[1]} columns")
except OSError as save_error:
    print(f"[ERROR] Failed to save cleaned dataset: {save_error}")
    raise


Cleaned dataset saved successfully to: C:\Users\Stalin\OneDrive\Desktop\draftrajproject\road-accident-severity-prediction\road-accident-severity-prediction\Dataset\processed\cleaned_accident_data.csv
Final shape: 2,715,940 rows x 38 columns


**Observations**

- The output directory is created programmatically with `Path.mkdir(parents=True,
  exist_ok=True)`, so the notebook runs successfully whether or not
  `Dataset/processed/` already exists.
- Saving with `index=False` avoids introducing a spurious unnamed index column into the
  cleaned CSV.


---
## 10. Final Summary

**Purpose:** Consolidate the full preprocessing pipeline into a single, clear summary
covering shape changes, columns removed, remaining missing values, the target variable,
and the output path.


In [17]:
original_shape = (accident_df.shape[0], accident_df.shape[1] + vehicle_df.shape[1] - 1)
final_shape = merged_df.shape
total_missing_remaining = int(merged_df.isnull().sum().sum())

final_summary = pd.DataFrame(
    {
        "Metric": [
            "Original Combined Shape (pre-cleaning, post-merge columns)",
            "Final Shape (rows x columns)",
            "Total Columns Removed",
            "Total Missing Values Remaining",
            "Target Variable",
            "Output Dataset Path",
        ],
        "Value": [
            f"{shape_after_merge[0]:,} rows x {shape_after_merge[1]} columns",
            f"{final_shape[0]:,} rows x {final_shape[1]} columns",
            (shape_after_merge[1] - final_shape[1]),
            total_missing_remaining,
            "Accident_Severity",
            str(OUTPUT_FILE.resolve()),
        ],
    }
)

final_summary


,Metric,Value
0,"Original Combined Shape (pre-cleaning, post-me...","2,715,940 rows x 57 columns"
1,Final Shape (rows x columns),"2,715,940 rows x 38 columns"
2,Total Columns Removed,19
3,Total Missing Values Remaining,1632422
4,Target Variable,Accident_Severity
5,Output Dataset Path,C:\Users\Stalin\OneDrive\Desktop\draftrajproje...


In [18]:
print("Columns removed during preprocessing:")
for col in columns_actually_removed:
    print(f"  - {col}")

print()
print(f"Columns dropped for excessive missingness: {len(dropped_for_missingness)}")
for col in dropped_for_missingness:
    print(f"  - {col}")


Columns removed during preprocessing:
  - Accident_Index
  - Vehicle_Reference
  - Number_of_Casualties
  - Location_Easting_OSGR
  - Location_Northing_OSGR
  - Police_Force
  - LSOA_of_Accident_Location
  - Local_Authority_(Highway)
  - 1st_Road_Number
  - 2nd_Road_Number
  - make
  - model

Columns dropped for excessive missingness: 7
  - 2nd_Road_Class
  - Carriageway_Hazards
  - Special_Conditions_at_Site
  - Driver_IMD_Decile
  - Hit_Object_in_Carriageway
  - Hit_Object_off_Carriageway
  - Skidding_and_Overturning


### Preprocessing Pipeline — Summary

This notebook transformed the two raw UK Road Safety files into a single,
analysis-ready dataset through the following pipeline:

1. **Merge:** `Accident_Information.csv` (base) was left-joined with
   `Vehicle_Information.csv` on `Accident_Index`, preserving every accident record
   (and therefore every target label) while expanding the dataset to vehicle-level
   granularity.
2. **Duplicate Removal:** Fully duplicated rows were identified and removed using a
   strict, all-column match.
3. **Missing Value Handling:** A transparent, threshold-based rule (low / moderate /
   high missingness) was applied uniformly across all columns — imputing where safe,
   preserving an explicit `"Unknown"` category where meaningful, and dropping columns
   only when missingness was too severe to reliably use.
4. **Column Removal:** Identifier, target-leakage, redundant, and low-value
   high-cardinality columns were removed, each with an explicit, printed justification.
5. **Data Type Correction:** Dates, times, integers, floats, and low-cardinality
   categorical columns were converted to semantically appropriate dtypes — with no
   encoding, scaling, or feature derivation performed.
6. **Consistency Checks:** The final dataset was verified to be free of duplicate rows
   and fully empty columns, to contain the target variable, and to have correctly
   dropped the merge key after use.
7. **Persistence:** The cleaned dataset was saved to
   `Dataset/processed/cleaned_accident_data.csv`, with the destination folder created
   automatically if needed.

### What This Notebook Deliberately Did *Not* Do

- No exploratory data analysis (distributions, correlations, visualizations).
- No feature engineering (no derived features, no date/time decomposition into
  additional columns beyond dtype correction).
- No encoding, scaling, or normalization of any column.
- No model training or evaluation.

### Next Steps

The next notebook (`03_Exploratory_Data_Analysis.ipynb`, Phase 4 of the project
roadmap) will use `Dataset/processed/cleaned_accident_data.csv` as its starting point to
explore distributions, relationships with `Accident_Severity`, and class imbalance —
still without performing feature engineering or modeling.
